# Dokumentenverarbeitung - Entwicklungsnotebook

Dieses Notebook zeigt die Entwicklung der PDF-Verarbeitungspipeline in zwei Phasen:

1. **Fruehes Prototyp** - erste Experimente mit direktem LangChain-Loader
2. **Finale Implementierung** - Nutzung des `src.processing`-Moduls


---
## Teil 1: Fruehes Prototyp

Erste Experimente zur PDF-Verarbeitung vor Entwicklung des `src.processing`-Moduls.
Dieser Code ist nicht mehr Teil der aktiven Pipeline.


In [ ]:
from pathlib import Path
import os

ROOT = Path(os.getcwd()).parent
PDF_FOLDER = ROOT / 'data' / 'raw'

pdf_files = list(PDF_FOLDER.rglob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'Keine PDFs gefunden in {PDF_FOLDER}')

print(f'{len(pdf_files)} PDF-Dateien gefunden:')
for f in pdf_files:
    print(' -', f)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def text_formatter(text: str) -> str:
    """Einfache Textbereinigung: mehrfache Zeilenumbrueche entfernen."""
    text = re.sub(r'+', ' ', text)
    return text.strip()

def read_pdf(pdf_path):
    """Laedt ein PDF und bereinigt den Text jeder Seite."""
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    for page in pages:
        page.page_content = text_formatter(page.page_content)
    return pages


In [ ]:
# Alle PDFs einlesen und zusammenfuehren
all_docs = []
for pdf_path in pdf_files:
    docs = read_pdf(pdf_path)
    all_docs.extend(docs)

print(f'{len(all_docs)} Seiten geladen')


In [ ]:
def chunk_documents(docs, chunk_size=1000, chunk_overlap=200):
    """Dokumente in kleinere Chunks aufteilen, Metadaten bleiben erhalten."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    return splitter.split_documents(docs)


In [ ]:
# Chunking testen
chunked_docs = chunk_documents(all_docs)
print(f'Anzahl erstellter Chunks: {len(chunked_docs)}')
print(chunked_docs[0].metadata)


In [ ]:
import re

# Muster fuer typische Kopf-/Fusszeilen aus DIN-Normen
DIN_HEADER_PATTERNS = [
    r'DEUTSCHE\s*NORM.*',
    r'Alleinverkauf.*Beuth\s+Verlag.*',
    r'Printed\s*copies\s*are\s*uncontrolled.*',
    r'Datum\s*/\s*Uhrzeit\s*des\s*Ausdrucks:.*',
    r'www\.din\.de|www\.beuth\.de',
    r'ICS\s*\d{2}\.\d{3}\.\d{2}',
]

def clean_din_text(text: str) -> str:
    """Bereinigt Zeilentrennungen und DIN-Kopfzeilen aus dem Text."""
    # Getrennte Woerter zusammenfuehren (z.B. 'Brems-gestell' -> 'Bremsgestell')
    text = re.sub(r'(\w)-\s*\s*(\w)', text)
    # Kopf-/Fusszeilen entfernen
    for pattern in DIN_HEADER_PATTERNS:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text.strip()


In [ ]:
# DIN-Bereinigung auf alle Seiten anwenden
for d in all_docs:
    d.page_content = clean_din_text(d.page_content)

print('Erste 500 Zeichen nach DIN-Bereinigung:')
print(all_docs[4].page_content[:500])


---
## Teil 2: Finale Implementierung mit `src.processing`

Nutzung des fertigen `src.processing`-Moduls mit `IngestConfig`,
automatischem Chunking und Deduplizierung.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)


In [ ]:
from datetime import datetime
import src.processing as processing

pdf_path = ROOT / 'data' / 'raw' / 'DZSF_2024_Sensorbasierte_Technologien.pdf'

cfg = processing.IngestConfig(
    skip_first_pages=2,
    min_chars=80,
    drop_toc=True,
    drop_noise_pages=True,
    chunk_size=1200,
    chunk_overlap=200,
)

pages, page_stats = processing.load_and_clean_pdf(pdf_path, cfg, verbose=True)
chunks, chunk_stats = processing.split_into_chunks(pages, cfg)

print(f'Seiten: {page_stats}')
print(f'Chunks: {chunk_stats}')

# Optional als JSONL speichern:
# run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
# processing.save_jsonl(pages, ROOT / 'data' / 'clean_pages' / f'pages_{run_id}.jsonl')
# processing.save_jsonl(chunks, ROOT / 'data' / 'chunks' / f'chunks_{run_id}.jsonl')


In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25-Retriever auf den erstellten Chunks testen
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 3

def zeige_treffer(query: str):
    ergebnisse = bm25.invoke(query)
    print(f'Anfrage: "{query}"')
    for i, doc in enumerate(ergebnisse, 1):
        src = doc.metadata.get('source', '?')
        page = doc.metadata.get('page', '?')
        print(f'  [{i}] {src}, Seite {page}')
        print(f'      {doc.page_content[:200]}...')
        print()

zeige_treffer('Bremszylinder Fuelldruck')
